In [1]:
!pip install unsloth datasets transformers trl peft accelerate bitsandbytes pyyaml -q
print("✅ Installation done")

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
✅ Installation done
CUDA available: True
GPU: AMD Instinct MI300X
VRAM: 206.1 GB


In [2]:
import json
from datasets import Dataset

# Path to your dataset (relative path from Notebooks folder)
file_path = "../Dataset/dataset-1.json"

# Load the JSON
with open(file_path, "r") as f:
    data = json.load(f)

print(f"✅ Loaded {len(data)} training examples")
print(f"Sample instruction: {data[0]['instruction'][:80]}...")

✅ Loaded 1035 training examples
Sample instruction: Send a Slack alert to the procurement team if a preferred supplier's contract is...


In [3]:
import os

# List files in Dataset folder
dataset_path = "../Dataset"
if os.path.exists(dataset_path):
    print("Files in Dataset folder:")
    for f in os.listdir(dataset_path):
        print(f"  - {f}")
else:
    print("Dataset folder not found at ../Dataset")
    
# Also check current directory
print("\nFiles in current directory:")
for f in os.listdir("."):
    if f.endswith(".json"):
        print(f"  - {f}")

Files in Dataset folder:
  - dataset-1.json
  - .ipynb_checkpoints
  - final-dataset-1.zip

Files in current directory:


In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-14B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

# Add LoRA adapters (explicitly set inference_mode=False)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", 
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
)

# Verify trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Trainable parameters: {trainable:,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    AMD Instinct MI300X. Num GPUs = 1. Max memory: 191.984 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+rocm7.2.4.git3d3aa833. ROCm Toolkit: 7.2.53211. Triton: 3.6.0+rocm7.2.4.git4ed88892
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

unsloth/qwen2.5-14b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.7 patched 48 layers with 48 QKV layers, 48 O layers and 48 MLP layers.


✅ Trainable parameters: 68,812,800


In [6]:
#old code before error............# # Loading the model
# from unsloth import FastLanguageModel
# import torch

# # Load base model (4-bit quantization to save VRAM)
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name="unsloth/Qwen2.5-14B-Instruct",
#     max_seq_length=2048,
#     load_in_4bit=True,
# )

# print("✅ Base model loaded successfully!")
# print(f"Model device: {model.device}")
# print(f"Model dtype: {model.dtype}")
# import json

# file_path = "../Dataset/dataset-1.json"

# with open(file_path, "r") as f:
#     data = json.load(f)
    

# print(f"Total examples: {len(data)}")

# print("\n--- First example ---")
# print("Instruction:", data[0].get("instruction", "MISSING")[:200])
# print("\nOutput (first 200 chars):", data[0].get("output", "MISSING")[:200])
# print("\n--- Second example ---")
# print("Instruction:", data[1].get("instruction", "MISSING")[:200])

==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    AMD Instinct MI300X. Num GPUs = 1. Max memory: 191.984 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+rocm7.2.4.git3d3aa833. ROCm Toolkit: 7.2.53211. Triton: 3.6.0+rocm7.2.4.git4ed88892
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

unsloth/qwen2.5-14b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Base model loaded successfully!
Model device: cuda:0
Model dtype: torch.bfloat16
Total examples: 1035

--- First example ---
Instruction: Send a Slack alert to the procurement team if a preferred supplier's contract is ending in less than 30 days.

Output (first 200 chars): pipeline:
  name: supplier_contract_expiration_alert
  description: Alerts procurement when a preferred supplier's contract is near expiration.
  version: "1.0"
  source:
    type: postgres_cdc
    co

--- Second example ---
Instruction: Compute the hourly count of active employees mapped by their store region and stream the summary to Snowflake for HR analytics.


In [5]:
!pip install scikit-learn -q
print("✅ scikit-learn installed")


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
✅ scikit-learn installed


In [6]:
# Splitting Data set
import json
from sklearn.model_selection import train_test_split

# Load your dataset
with open("../Dataset/dataset-1.json", "r") as f:
    data = json.load(f)

# Split: 80% train, 20% validation
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)

print(f"✅ Total examples: {len(data)}")
print(f"✅ Training examples: {len(train_data)}")
print(f"✅ Validation examples: {len(val_data)}")

✅ Total examples: 1035
✅ Training examples: 828
✅ Validation examples: 207


testing with simple Query...

Earlier Used to got error.. Error is about..
    It is fantastic that you finally got a clean result! Since you are working on top-tier enterprise hardware (AMD Instinct MI300X with 206 GB VRAM), understanding exactly why this happened will save you days of troubleshooting during your upcoming fine-tuning and deployment stages.
The crash was caused by a specific conflict between Hugging Face’s automated text generation wrapper, Unsloth’s optimization layers, and the AMD ROCm driver ecosystem. [1] 
Here is the step-by-step breakdown of why that error occurred:
## 1. The Core Culprit: The max_length=32768 Conflict
When you load a modern 14B model like Qwen, its internal configuration defaults to a massive context window (usually 32,768 tokens).

* When you called model.generate(max_new_tokens=150), Hugging Face's internal code tried to automatically build a static, massive dynamic-allocation matrix behind the scenes based on that max_length setting.
* Under the hood, this triggers an internal Hugging Face warning. On NVIDIA/CUDA hardware, this warning is harmless. However, on AMD ROCm, it behaves differently due to compiler mechanics.

## 2. The ROCm AOTriton Kernel Compilation Bug
The warning you saw right before the crash was always followed by an unhandled system signal (usually a Segmentation Fault / SIGSEGV). [2] 

* What Hugging Face did: Because of the token/length confusion and a shared Pad/EOS token ID, Hugging Face tried to dynamically generate a specific type of boolean attention mask matrix.
* What ROCm did: To run fast on your MI300X, Unsloth uses AOTriton (Ahead-of-Time Triton compiler) or FlashAttention kernels mapped specifically for AMD's Matrix Core architecture.
* The Collision: When Hugging Face passed that poorly formed, auto-generated attention mask into the optimized AMD mathematical kernels, the compiler encountered an indexing mismatch. Instead of throwing a clean Python exception, it caused a low-level C++ memory access violation inside the AMD ROCm driver. The operating system intervened and instantly killed the Python process to protect the GPU—which is why your Jupyter kernel simply died without a trace. [3] 

## 3. Why the Direct Tensor Math Fixed It
By rewriting the execution to use a manual loop (outputs = model(input_ids=input_ids)), we completely bypassed Hugging Face's complex generation wrapper.

* We completely ignored the max_length vs max_new_tokens configuration parser.
* We didn't pass any auto-generated attention masks.
* The MI300X was forced to perform simple, raw matrix multiplication on the tokens one by one, which the AMD driver can execute flawlessly.

## What this means for your next steps (Fine-Tuning):
Now that you know the model weights are perfectly healthy, you need to be careful when moving to the training phase.
When you configure your fine-tuning script (using Unsloth or SFTTrainer), make sure to explicitly set a reasonable training max sequence length (like max_seq_length=2048 or 4096) right from the start. Never let the training arguments default to the model's absolute maximum of 32k, or you risk triggering the same driver-level matrix allocation crashes.
Are you ready to look at how to format your training arguments safely for ROCm, or would you like to verify your dataset processing pipeline first?

[1] [https://learn.microsoft.com](https://learn.microsoft.com/en-us/answers/questions/4064970/how-do-i-fix-video-tdr-failure%28116%29)
[2] [https://blenderartists.org](https://blenderartists.org/t/too-many-crashes/1526503)
[3] [https://root-forum.cern.ch](https://root-forum.cern.ch/t/jupyter-notebook-with-root-c-kernel/24399)


In [3]:
import os
import torch
import warnings

# 1. Enforce strict background configurations
warnings.filterwarnings("ignore")
os.environ["TORCH_ROCM_AOTRITON_ENABLE_CACHE"] = "1"

# 2. Setup prompt and clean inputs
test_instruction = "Do You know about india"
print("="*60)
print("TEST INSTRUCTION:")
print("="*60)
print(test_instruction)

prompt = f"### Instruction:\n{test_instruction}\n\n### Response:\n"

# Tokenize cleanly without background parameters
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
input_ids = inputs.input_ids

print("\nRunning direct tensor-math generation (Bypassing .generate())...")

# 3. Manual sequence execution loop (Generates 50 tokens step-by-step safely)
generated_tokens = []
max_tokens_to_generate = 50

with torch.no_grad():
    for _ in range(max_tokens_to_generate):
        # Forward pass through the raw model architecture
        outputs = model(input_ids=input_ids)
        
        # Pull the logit scores for the absolute last token generated
        next_token_logits = outputs.logits[:, -1, :]
        
        # Filter with a safe temperature setup directly on the logits matrix
        probs = torch.softmax(next_token_logits / 0.3, dim=-1)
        
        # Sample the next token token ID
        next_token = torch.multinomial(probs, num_samples=1)
        
        # Append to our tracker
        generated_tokens.append(next_token.item())
        
        # Break execution instantly if the model outputs an End of Sequence token
        if next_token.item() == tokenizer.eos_token_id:
            break
            
        # Cat the new token to the sequence so the next loop has full context
        input_ids = torch.cat([input_ids, next_token], dim=-1)

# 4. Decode the manual array
response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("\n" + "="*60)
print("BASE MODEL OUTPUT (Raw Tensor Evaluation):")
print("="*60)
print(response)


TEST INSTRUCTION:
Do You know about india

Running direct tensor-math generation (Bypassing .generate())...

BASE MODEL OUTPUT (Raw Tensor Evaluation):
Yes, I know about India. It is the world's seventh-largest country by area, the second-most populous country with over 1.3 billion people, and the most populous democracy in the world. India is a federal republic with a parliamentary system


Formatting Dataset

In [8]:
def format_examples(example):
    text = f"""### Instruction:
{example["instruction"]}

### Response:
{example["output"]}
"""
    return {"text": text}

from datasets import Dataset

train_dataset = Dataset.from_list(train_data).map(format_examples)
val_dataset = Dataset.from_list(val_data).map(format_examples)

print(f"✅ Train: {len(train_dataset)} | Validation: {len(val_dataset)}")
print("\n📝 Sample formatted training example:")
print(train_dataset[0]["text"][:400])

Map:   0%|          | 0/828 [00:00<?, ? examples/s]

Map:   0%|          | 0/207 [00:00<?, ? examples/s]

✅ Train: 828 | Validation: 207

📝 Sample formatted training example:
### Instruction:
Alert the logistics team on Slack if a shipment item has a quantity_shipped of exactly 50.

### Response:
pipeline:
  name: specific_bulk_shipment_item_alert
  version: "1.0"
  source:
    type: postgres_cdc
    connection: megastore_primary
    tables:
      - table: shipment_items
        primary_key: shipment_item_id
        capture_mode: cdc_only
    kafka_topic: megastore.ret


 Run Training.........................................

In [10]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    args=TrainingArguments(
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=10,
        eval_steps=50,
        save_steps=100,
        eval_strategy="steps",          # ← Fixed: was evaluation_strategy
        output_dir="./outputs",
        fp16=False,
        bf16=True,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        report_to="none",
    ),
)

print("🚀 Starting training...")
trainer.train()
print("✅ Training complete!")

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/828 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/207 [00:00<?, ? examples/s]

ValueError: You cannot perform fine-tuning on purely quantized models. Please attach trainable adapters on top of the quantized model to correctly perform fine-tuning. Please see: https://huggingface.co/docs/transformers/peft for more details

In [9]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=10,
        output_dir="./outputs",
        fp16=False,
        bf16=True,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        report_to="none",
    ),
)

print("🚀 Trainer ready. Call trainer.train() to start.")

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/828 [00:00<?, ? examples/s]

🚀 Trainer ready. Call trainer.train() to start.


In [10]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 828 | Num Epochs = 3 | Total steps = 621
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 68,812,800 of 14,838,846,464 (0.46% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.506811
20,0.562123
30,0.361779
40,0.268734
50,0.249797
60,0.298932
70,0.248745
80,0.250073
90,0.242635
100,0.210071


Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-500/tokenizer_config.json.


PicklingError: Can't pickle <class 'trl.trainer.sft_config.SFTConfig'>: it's not the same object as trl.trainer.sft_config.SFTConfig

In [11]:
# Save your trained model (bypasses the pickling error)
model.save_pretrained("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")
print("✅ Model saved to ./fine_tuned_model")

Unsloth: Restored added_tokens_decoder metadata in ./fine_tuned_model/tokenizer_config.json.


✅ Model saved to ./fine_tuned_model


In [ ]:
# Below successfully Tested..

In [13]:
# # Reuse the manual loop that worked for the base model
# test_instruction = "Alert the logistics team on Slack if a shipment item has a quantity_shipped of exactly 50."
# prompt = f"### Instruction:\n{test_instruction}\n\n### Response:\n"

# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# input_ids = inputs.input_ids

# print("Manual generation on fine-tuned model...")
# generated_tokens = []
# max_tokens = 150

# with torch.no_grad():
#     for _ in range(max_tokens):
#         outputs = model(input_ids=input_ids)
#         next_logits = outputs.logits[:, -1, :]
#         probs = torch.softmax(next_logits / 0.3, dim=-1)
#         next_token = torch.multinomial(probs, num_samples=1)
#         if next_token.item() == tokenizer.eos_token_id:
#             break
#         generated_tokens.append(next_token.item())
#         input_ids = torch.cat([input_ids, next_token], dim=-1)

# response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
# print("="*60)
# print("FINE-TUNED MODEL OUTPUT (manual loop):")
# print("="*60)
# print(response)

Manual generation on fine-tuned model...
FINE-TUNED MODEL OUTPUT (manual loop):
pipeline:
  name: exact_50_shipment_item_alert
  version: "1.0"
  source:
    type: postgres_cdc
    connection: megastore_primary
    tables:
      - table: shipment_items
        primary_key: shipment_item_id
        capture_mode: cdc_only
    kafka_topic: megastore.retail_events
  transforms:
    - id: filter_exact_50
      type: filter
      source: shipment_items
      conditions:
        - field: quantity_shipped
          operator: EQUALS
          value: 50
  sink:
    - id: slack_logistics
      type: slack
      source: filter_exact_50



In [15]:
def safe_generate(model, tokenizer, prompt, max_new_tokens=200, temperature=0.1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_ids = inputs.input_ids
    generated_tokens = []
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(input_ids=input_ids)
            next_logits = outputs.logits[:, -1, :]
            if temperature > 0:
                probs = torch.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = torch.argmax(next_logits, dim=-1, keepdim=True)
            if next_token.item() == tokenizer.eos_token_id:
                break
            generated_tokens.append(next_token.item())
            input_ids = torch.cat([input_ids, next_token], dim=-1)
    
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

# Test it on your instruction
prompt = """### Instruction:
Alert the logistics team on Slack if a shipment item has a quantity_shipped of exactly 50.

### Response:
"""
output = safe_generate(model, tokenizer, prompt, max_new_tokens=200)
print("="*60)
print("GENERATED YAML (via safe_generate):")
print("="*60)
print(output)

GENERATED YAML (via safe_generate):
pipeline:
  name: exact_50_shipped_alert
  version: "1.0"
  source:
    type: postgres_cdc
    connection: megastore_primary
    tables:
      - table: shipment_items
        primary_key: shipment_item_id
        capture_mode: cdc_only
    kafka_topic: megastore.retail_events
  transforms:
    - id: filter_exact_50
      type: filter
      source: shipment_items
      conditions:
        - field: quantity_shipped
          operator: EQUALS
          value: 50
  sink:
    - id: slack_logistics
      type: slack
      source: filter_exact_50
      channel: "#logistics-alerts"
      webhook_url: "${env.SLACK_WEBHOOK_LOGISTICS}"
      message_template: |
        :truck: *Exact 50 Shipped*
        Shipment Item ID: ${shipment_item_id}


# Testing with datset and finetuned model... calculating accuracy...........

In [21]:
import yaml

def is_valid_yaml(text):
    try:
        # Remove any extra text before/after YAML block
        yaml_part = text.strip()
        # Try to parse
        parsed = yaml.safe_load(yaml_part)
        # Check if it contains pipeline structure
        if not isinstance(parsed, dict):
            return False
        # Basic required keys (adapt to your YAML structure)
        required = {"name", "source", "sinks"}
        # If pipeline is nested under "pipeline" key
        if "pipeline" in parsed:
            parsed = parsed["pipeline"]
        return any(key in parsed for key in required)
    except:
        return False

# Test on a few validation examples
test_samples = val_data[:50]  # use 50 samples for quick eval
correct = 0

for ex in test_samples:
    prompt = f"### Instruction:\n{ex['instruction']}\n\n### Response:\n"
    generated = safe_generate(model, tokenizer, prompt, max_new_tokens=200)
    if is_valid_yaml(generated):
        correct += 1

accuracy = correct / len(test_samples)
print(f"Accuracy on {len(test_samples)} samples: {accuracy:.2%}")

Accuracy on 50 samples: 92.00%


In [27]:
print(accuracy)

0.92


traing on remained dataset and ..... checkpoints...

In [17]:
# Recreate the trainer exactly as before, but with resume_from_checkpoint
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=10,
        output_dir="./outputs",
        fp16=False,
        bf16=True,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        report_to="none",
        save_steps=50,
    ),
)

# Resume from the last checkpoint
trainer.train(resume_from_checkpoint="./outputs/checkpoint-500")

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/828 [00:00<?, ? examples/s]

FileNotFoundError: [Errno 2] No such file or directory: './outputs/checkpoint-500/trainer_state.json'

In [22]:
import os
import time

# Check if the output directory has been updated recently
output_dir = "./outputs"
if os.path.exists(output_dir):
    latest = max(os.listdir(output_dir), key=lambda x: os.path.getmtime(os.path.join(output_dir, x)))
    print(f"Latest checkpoint: {latest}")
    print(f"Modified: {time.ctime(os.path.getmtime(os.path.join(output_dir, latest)))}")
else:
    print("No outputs yet")

Latest checkpoint: checkpoint-500
Modified: Mon Jun 15 19:28:44 2026


In [23]:
import os

output_dir = "./outputs"
if os.path.exists(output_dir):
    
    items = os.listdir(output_dir)
    print("Contents of outputs:")
    for item in items:
        item_path = os.path.join(output_dir, item)
        if os.path.isdir(item_path):
            print(f"  [DIR] {item}")
        else:
            print(f"  [FILE] {item}")
else:
    print("outputs directory not found")

Contents of outputs:
  [DIR] checkpoint-500
  [FILE] README.md


In [24]:
import os
checkpoint_path = "./outputs/checkpoint-500"
if os.path.exists(checkpoint_path):
    files = os.listdir(checkpoint_path)
    print("Files in checkpoint-500:")
    for f in files:
        print(f"  {f}")
else:
    print("checkpoint-500 not found")

Files in checkpoint-500:
  adapter_config.json
  adapter_model.safetensors
  training_args.bin
  README.md
  tokenizer.json
  chat_template.jinja
  tokenizer_config.json


In [ ]:
################################################

saving model with 92 percent accuracy on 500 QA pairs.......................

In [28]:
# Create a folder for the final model
final_model_dir = "./final_model"
model.save_pretrained(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print(f"✅ Model saved to {final_model_dir}")

# Save the safe_generate function as a Python file (for easy import)
with open("safe_generate.py", "w") as f:
    f.write("""
import torch

def safe_generate(model, tokenizer, prompt, max_new_tokens=200, temperature=0.1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_ids = inputs.input_ids
    generated_tokens = []
    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(input_ids=input_ids)
            next_logits = outputs.logits[:, -1, :]
            if temperature > 0:
                probs = torch.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = torch.argmax(next_logits, dim=-1, keepdim=True)
            if next_token.item() == tokenizer.eos_token_id:
                break
            generated_tokens.append(next_token.item())
            input_ids = torch.cat([input_ids, next_token], dim=-1)
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)
""")
print("✅ safe_generate.py saved")

Unsloth: Restored added_tokens_decoder metadata in ./final_model/tokenizer_config.json.


✅ Model saved to ./final_model
✅ safe_generate.py saved
